# Experiment 3: Layered Failure Analysis

L1-L5 failure distribution, layer-conditioned fragility, CF-exposed failures.

**Core thesis claim**: AL failures concentrate in L3 (Event-driven), L4 (Workflow), L5 (Toolchain).

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from utils import load_all_results

from bcbench.analysis.aggregator import build_families
from bcbench.analysis.family import FamilyOutcome
from bcbench.analysis.metrics import (
    cf_exposed_failure_count,
    failure_layer_distribution,
    layer_conditioned_fragility,
)
from bcbench.results.base import ExecutionBasedEvaluationResult
from bcbench.types import FailureLayer

LAYER_ORDER = [layer.value for layer in FailureLayer]
LAYER_SHORT = {layer.value: layer.name.split("_")[0] for layer in FailureLayer}

# 3 CFE runs
SETUPS = ["copilot-cf-opus-4-6", "copilot-cf-sonnet-4-6", "copilot-cf-gpt-5-4", "copilot-cf-gpt-5-3-codex"]
SETUP_LABELS = {
    "copilot-cf-opus-4-6": "Claude Opus 4.6",
    "copilot-cf-sonnet-4-6": "Claude Sonnet 4.6",
    "copilot-cf-gpt-5-4": "GPT-5-4",
    "copilot-cf-gpt-5-3-codex": "GPT-5-3 Codex",
}

all_results = load_all_results(category="bug-fix")


def df_to_results(df: pd.DataFrame) -> list[ExecutionBasedEvaluationResult]:
    results = []
    for _, row in df.iterrows():
        results.append(
            ExecutionBasedEvaluationResult(
                instance_id=row["instance_id"],
                project=row.get("project", ""),
                model=row.get("model", "unknown"),
                agent_name="GitHub Copilot",
                category="cf",
                resolved=bool(row.get("resolved", False)),
                build=bool(row.get("build", False)),
                output=row.get("output", ""),
            )
        )
    return results


# Post-hoc failure-layer labels: LLM-annotated by GPT-5.3-Codex following the
# annotation handbook (notebooks/counterfactual-evaluation/ANNOTATION_HANDBOOK.md).
# Labels are per (model, instance) — a base and its CFs can fail at different layers.
_codex_path = Path.cwd() / "codex_failure_layers_thesis.jsonl"
if not _codex_path.exists():
    _codex_path = Path.cwd() / "counterfactual-evaluation" / "codex_failure_layers_thesis.jsonl"

_VALID = {layer.value for layer in FailureLayer}
instance_layers_by_model: dict[str, dict[str, FailureLayer]] = {}
if _codex_path.exists():
    for _line in _codex_path.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if not _line:
            continue
        _o = json.loads(_line)
        if _o["primary_failure_layer"] in _VALID:  # skip UNCERTAIN
            instance_layers_by_model.setdefault(_o["model"], {})[_o["instance_id"]] = FailureLayer(_o["primary_failure_layer"])


def _base_id(instance_id: str) -> str:
    return instance_id.split("__cf-")[0]


def _family_layers(inst_layers: dict[str, FailureLayer]) -> dict[str, FailureLayer]:
    """Family-level map (base_id -> layer): keep the most fundamental (lowest) layer in the family."""
    order = list(FailureLayer)
    out: dict[str, FailureLayer] = {}
    for iid, lay in inst_layers.items():
        b = _base_id(iid)
        if b not in out or order.index(lay) < order.index(out[b]):
            out[b] = lay
    return out


families_by_model: dict[str, list[FamilyOutcome]] = {}
for setup in SETUPS:
    if setup not in all_results:
        print(f"WARNING: {setup} not found")
        continue
    label = SETUP_LABELS.get(setup, setup)
    results = df_to_results(all_results[setup])
    families = build_families(results, failure_layers=_family_layers(instance_layers_by_model.get(label, {})))
    families_by_model[label] = families
    print(f"{label}: {len(families)} families, {len(instance_layers_by_model.get(label, {}))} layer-labeled failed instances")

# Per-instance long table (the faithful unit for layer distribution: one row per failed output)
_inst_rows = []
for model, inst_layers in instance_layers_by_model.items():
    for iid, lay in inst_layers.items():
        _inst_rows.append({
            "Model": model,
            "instance_id": iid,
            "Kind": "CF" if "__cf-" in iid else "base",
            "Layer": LAYER_SHORT[lay.value],
            "layer_value": lay.value,
        })
instance_layer_df = pd.DataFrame(_inst_rows)

print(f"\nReady. {len(families_by_model)} models, {len(instance_layer_df)} labeled failed instances.")

Claude Opus 4.6: 71 families, 45 layer-labeled failed instances
Claude Sonnet 4.6: 71 families, 44 layer-labeled failed instances
GPT-5-4: 71 families, 42 layer-labeled failed instances


GPT-5-3 Codex: 71 families, 48 layer-labeled failed instances

Ready. 4 models, 179 labeled failed instances.


## L1-L5 Failure Distribution

In [2]:
def plot_failure_layer_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        dist = failure_layer_distribution(families)
        total = sum(dist.values()) or 1
        for layer in LAYER_ORDER:
            count = dist.get(layer, 0)
            rows.append(
                {
                    "Model": model,
                    "Layer": LAYER_SHORT.get(layer, layer),
                    "Count": count,
                    "Proportion (%)": round(count / total * 100, 1),
                }
            )

    df = pd.DataFrame(rows)
    fig = px.bar(
        df,
        x="Layer",
        y="Proportion (%)",
        color="Model",
        barmode="group",
        title="Failure Layer Distribution (L1-L5) by Model",
        text_auto=True,
        category_orders={"Layer": ["L1", "L2", "L3", "L4", "L5"]},
    )
    fig.show()
    return df


print("plot_failure_layer_distribution() ready")

plot_failure_layer_distribution() ready


## Layer-Conditioned Fragility Heatmap

Fragile family % per layer per model — the core diagnostic matrix.

In [3]:
def plot_layer_conditioned_fragility(families_by_model: dict[str, list[FamilyOutcome]]):
    models = list(families_by_model.keys())
    layers = ["L1", "L2", "L3", "L4", "L5"]
    matrix = []

    for model in models:
        lcf = layer_conditioned_fragility(families_by_model[model])
        row = [round(lcf.get(layer, 0) * 100, 1) for layer in LAYER_ORDER]
        matrix.append(row)

    fig = go.Figure(
        data=go.Heatmap(
            z=matrix,
            x=layers,
            y=models,
            colorscale="RdYlGn_r",
            text=matrix,
            texttemplate="%{text}%",
            colorbar_title="Fragility %",
        )
    )
    fig.update_layout(
        title="Layer-Conditioned Fragility: P(CF fail | Base correct) per Layer",
        xaxis_title="Failure Layer",
        yaxis_title="Model",
    )
    fig.show()


print("plot_layer_conditioned_fragility() ready")

plot_layer_conditioned_fragility() ready


## Layer-Conditioned Severity

In [4]:
def plot_severity_by_layer(families: list[FamilyOutcome]):
    rows = []
    for f in families:
        if f.failure_layer and f.severity is not None:
            rows.append(
                {
                    "Layer": LAYER_SHORT.get(f.failure_layer.value, f.failure_layer.value),
                    "Severity": f.severity,
                    "Type": f.family_type.value,
                }
            )

    df = pd.DataFrame(rows)
    if not df.empty:
        fig = px.box(
            df,
            x="Layer",
            y="Severity",
            color="Layer",
            title="Fragility Severity by Failure Layer",
            points="all",
            category_orders={"Layer": ["L1", "L2", "L3", "L4", "L5"]},
        )
        fig.show()
    return df


print("plot_severity_by_layer() ready")

plot_severity_by_layer() ready


## CF-Exposed Failure Analysis

Families where failure ONLY appears in CF (base correct, all CFs fail).

In [5]:
def analyze_cf_exposed(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        total = len(families)
        exposed = cf_exposed_failure_count(families)
        fragile = sum(1 for f in families if f.is_fragile)
        rows.append(
            {
                "Model": model,
                "Total Families": total,
                "Fragile": fragile,
                "CF-Exposed (all CFs fail)": exposed,
                "CF-Exposed %": round(exposed / total * 100, 1) if total else 0,
            }
        )

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


print("analyze_cf_exposed() ready")

analyze_cf_exposed() ready


## Per-Instance Failure Layer Distribution (post-hoc, Codex-annotated)

In [6]:
def plot_instance_layer_distribution(instance_layer_df: pd.DataFrame):
    """Layer distribution over individual failed outputs (not families)."""
    order = ["L1", "L2", "L3", "L4", "L5"]
    if instance_layer_df.empty:
        print("No labeled instances.")
        return instance_layer_df

    by_model = (
        instance_layer_df.groupby(["Model", "Layer"]).size().reset_index(name="Count")
    )
    fig = px.bar(
        by_model, x="Layer", y="Count", color="Model", barmode="group",
        title="Failure Layer Distribution per Failed Output, by Model",
        category_orders={"Layer": order}, text_auto=True,
    )
    fig.show()

    by_kind = instance_layer_df.groupby(["Kind", "Layer"]).size().reset_index(name="Count")
    fig2 = px.bar(
        by_kind, x="Layer", y="Count", color="Kind", barmode="group",
        title="Failure Layer Distribution: base vs CF",
        category_orders={"Layer": order, "Kind": ["base", "CF"]}, text_auto=True,
    )
    fig2.show()

    pivot = instance_layer_df.pivot_table(index="Model", columns="Layer", values="instance_id", aggfunc="count", fill_value=0)
    return pivot.reindex(columns=[c for c in order if c in pivot.columns])


print("plot_instance_layer_distribution() ready")

plot_instance_layer_distribution() ready


## Inter-Annotator Agreement (post-hoc layer labels)

Annotator A = GPT-5.3-Codex (the canonical `codex_failure_layers_thesis.jsonl`); annotator B = an independent Claude Opus 4.8 pass over a stratified 70-instance double-annotation subset (all non-L4 labels plus a random L4 sample, so the hard L2/L3/L4 boundaries are stressed). Agreement is computed with `bcbench.analysis.annotation.inter_annotator_agreement`.

In [7]:
from bcbench.analysis.annotation import inter_annotator_agreement

def _load_labels(fname, key="primary_failure_layer"):
    out = {}
    p = Path.cwd() / fname
    if not p.exists():
        p = Path.cwd() / "counterfactual-evaluation" / fname
    if not p.exists():
        return out
    for line in p.read_text(encoding="utf-8").splitlines():
        if line.strip():
            o = json.loads(line)
            out[(o["model"], o["instance_id"])] = o[key]
    return out

_A = _load_labels("codex_failure_layers_thesis.jsonl")
_B = _load_labels("iaa_annotator_b.jsonl")
_keys = sorted(set(_A) & set(_B))
if _keys:
    a = {f"{m}|{i}": _A[(m, i)] for (m, i) in _keys}
    b = {f"{m}|{i}": _B[(m, i)] for (m, i) in _keys}
    iaa = inter_annotator_agreement(a, b)
    print(f"IAA (codex A vs Opus-4.8 B): n={iaa.n}  agreement={iaa.agreement_rate:.3f}  Cohen kappa={iaa.cohen_kappa:.3f}")

    conf = (
        pd.DataFrame({"A": [LAYER_SHORT.get(a[k], a[k]) for k in a],
                      "B": [LAYER_SHORT.get(b[k], b[k]) for k in a]})
        .pivot_table(index="A", columns="B", aggfunc=len, fill_value=0)
    )
    display(conf)
else:
    print("Second-annotator file iaa_annotator_b.jsonl not found; skipping IAA.")

IAA (codex A vs Opus-4.8 B): n=70  agreement=0.643  Cohen kappa=0.525


B,L1,L2,L3,L4,L5
A,,,,,
L1,8,0,0,5,0
L2,0,1,0,8,0
L3,0,0,6,11,0
L4,0,0,0,20,0
L5,1,0,0,0,10


## Run All Analysis

In [8]:
# Layer distribution (needs failure_layers populated)
layer_df = plot_failure_layer_distribution(families_by_model)
display(layer_df)

# Layer-conditioned fragility heatmap
plot_layer_conditioned_fragility(families_by_model)

# CF-exposed failures
cf_df = analyze_cf_exposed(families_by_model)
display(cf_df)

# Per-instance (post-hoc Codex-annotated) layer distribution
inst_layer_pivot = plot_instance_layer_distribution(instance_layer_df)
display(inst_layer_pivot)

,Model,Layer,Count,Proportion (%)
0,Claude Opus 4.6,L1,1,3.2
1,Claude Opus 4.6,L2,1,3.2
2,Claude Opus 4.6,L3,4,12.9
3,Claude Opus 4.6,L4,23,74.2
4,Claude Opus 4.6,L5,2,6.5
5,Claude Sonnet 4.6,L1,1,3.2
6,Claude Sonnet 4.6,L2,3,9.7
7,Claude Sonnet 4.6,L3,3,9.7
8,Claude Sonnet 4.6,L4,24,77.4
9,Claude Sonnet 4.6,L5,0,0.0


            Model  Total Families  Fragile  CF-Exposed (all CFs fail)  CF-Exposed %
  Claude Opus 4.6              71       15                          9          12.7
Claude Sonnet 4.6              71       15                          7           9.9
          GPT-5-4              71       13                          8          11.3
    GPT-5-3 Codex              71       13                          7           9.9


,Model,Total Families,Fragile,CF-Exposed (all CFs fail),CF-Exposed %
0,Claude Opus 4.6,71,15,9,12.7
1,Claude Sonnet 4.6,71,15,7,9.9
2,GPT-5-4,71,13,8,11.3
3,GPT-5-3 Codex,71,13,7,9.9


Layer,L1,L2,L3,L4,L5
Model,,,,,
Claude Opus 4.6,2,1,4,35,3
Claude Sonnet 4.6,1,4,4,35,0
GPT-5-3 Codex,4,2,5,33,4
GPT-5-4,6,2,4,26,4
